# NTL Detection: Head-to-Head Comparison

Loads saved results from the individual approach notebooks and generates side-by-side comparisons.

**Prerequisites:** Run at least Approaches A and B first. Approach C is optional (requires Chronos-2 endpoint).

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    auc as sklearn_auc,
    classification_report, confusion_matrix,
    precision_recall_curve, precision_recall_fscore_support, roc_auc_score,
)

from utils import load_results, RESULTS_DIR

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("Imports OK")

In [ ]:
# Load available results
approaches = {}
labels = {"approach-a": "A: Baseline", "approach-b": "B: Enhanced", "approach-c": "C: Forecast+Classify"}
colors = {"approach-a": "#3498db", "approach-b": "#f39c12", "approach-c": "#2ecc71"}

for key, name in labels.items():
    path = RESULTS_DIR / f"{key}.json"
    if path.exists():
        y_test, y_pred, y_prob = load_results(key)
        approaches[key] = {"name": name, "y_test": y_test, "y_pred": y_pred, "y_prob": y_prob}
        print(f"Loaded {name}")
    else:
        print(f"Skipped {name} (no results file)")

print(f"\n{len(approaches)} approaches loaded for comparison.")

In [ ]:
# Compute metrics at default threshold (0.5) and F1-optimal threshold
import pandas as pd

results = {}
results_optimal = {}

for key, data in approaches.items():
    y_test, y_pred, y_prob = data["y_test"], data["y_pred"], data["y_prob"]

    # Default threshold (0.5)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary")
    auc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    fp, tp, fn, tn = cm[0, 1], cm[1, 1], cm[1, 0], cm[0, 0]
    results[data["name"]] = {
        "Precision": p, "Recall": r, "F1": f1, "ROC AUC": auc,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn, "Threshold": 0.5,
    }

    # F1-optimal threshold
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_idx = np.argmax(f1s)
    best_thresh = thresholds[best_idx]
    y_pred_opt = (y_prob >= best_thresh).astype(int)
    p_opt, r_opt, f1_opt, _ = precision_recall_fscore_support(y_test, y_pred_opt, average="binary")
    cm_opt = confusion_matrix(y_test, y_pred_opt)
    fp_opt, tp_opt = cm_opt[0, 1], cm_opt[1, 1]
    results_optimal[data["name"]] = {
        "Precision": p_opt, "Recall": r_opt, "F1": f1_opt, "ROC AUC": auc,
        "TP": tp_opt, "FP": fp_opt, "Threshold": best_thresh,
    }

results_df = pd.DataFrame(results).T
results_opt_df = pd.DataFrame(results_optimal).T

print("=" * 80)
print("COMPARISON AT DEFAULT THRESHOLD (0.5)")
print("=" * 80)
print(results_df[["Precision", "Recall", "F1", "ROC AUC", "FP", "TP"]].to_string(float_format="{:.4f}".format))

print("\n" + "=" * 80)
print("COMPARISON AT F1-OPTIMAL THRESHOLD (post-hoc, no retraining)")
print("=" * 80)
print(results_opt_df[["Threshold", "Precision", "Recall", "F1", "ROC AUC", "FP", "TP"]].to_string(float_format="{:.4f}".format))

## Visual Comparison

In [ ]:
approach_names = list(results.keys())
approach_colors = [colors[k] for k in approaches.keys()]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# --- Top row: Default threshold (0.5) ---

# Bar chart: Precision, Recall, F1 at default threshold
ax = axes[0, 0]
x = np.arange(len(approach_names))
width = 0.25
metric_colors = ["#3498db", "#f39c12", "#2ecc71"]
for i, metric in enumerate(["Precision", "Recall", "F1"]):
    vals = [results[name][metric] for name in approach_names]
    ax.bar(x + i * width, vals, width, label=metric, color=metric_colors[i], alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(approach_names, fontsize=9)
ax.set_ylabel("Score")
ax.set_ylim(0, 0.8)
ax.set_title("Default Threshold (0.5)")
ax.legend()
for i, metric in enumerate(["Precision", "Recall", "F1"]):
    vals = [results[name][metric] for name in approach_names]
    for j, v in enumerate(vals):
        ax.text(j + i * width, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)

# Precision-Recall curves
ax = axes[0, 1]
for (key, data), color in zip(approaches.items(), approach_colors):
    precision_vals, recall_vals, _ = precision_recall_curve(data["y_test"], data["y_prob"])
    auc_pr = sklearn_auc(recall_vals, precision_vals)
    ax.plot(recall_vals, precision_vals, label=f"{data['name']} (AUPRC={auc_pr:.3f})", color=color, linewidth=2)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend(loc="best")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# --- Bottom row: F1-optimal threshold ---

# Bar chart: Precision, Recall, F1 at optimal threshold
ax = axes[1, 0]
for i, metric in enumerate(["Precision", "Recall", "F1"]):
    vals = [results_optimal[name][metric] for name in approach_names]
    ax.bar(x + i * width, vals, width, label=metric, color=metric_colors[i], alpha=0.85)
ax.set_xticks(x + width)
xlabels = [f"{name}\n(t={results_optimal[name]['Threshold']:.3f})" for name in approach_names]
ax.set_xticklabels(xlabels, fontsize=9)
ax.set_ylabel("Score")
ax.set_ylim(0, 0.8)
ax.set_title("F1-Optimal Threshold (post-hoc)")
ax.legend()
for i, metric in enumerate(["Precision", "Recall", "F1"]):
    vals = [results_optimal[name][metric] for name in approach_names]
    for j, v in enumerate(vals):
        ax.text(j + i * width, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)

# False positives comparison: default vs optimal
ax = axes[1, 1]
bar_width = 0.35
fp_default = [results[name]["FP"] for name in approach_names]
fp_optimal = [results_optimal[name]["FP"] for name in approach_names]
bars1 = ax.bar(x - bar_width / 2, fp_default, bar_width, label="Default (0.5)", color=approach_colors, alpha=0.5)
bars2 = ax.bar(x + bar_width / 2, fp_optimal, bar_width, label="F1-Optimal", color=approach_colors, alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(approach_names, fontsize=9)
ax.set_ylabel("False Positives (wasted inspections)")
ax.set_title("False Positives: Default vs F1-Optimal Threshold")
ax.legend()
for bar, count in zip(bars1, fp_default):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, str(int(count)),
            ha="center", fontsize=9, alpha=0.6)
for bar, count in zip(bars2, fp_optimal):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, str(int(count)),
            ha="center", fontsize=9, fontweight="bold")

plt.suptitle("NTL Detection: Approach Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved to comparison_results.png")

## Operational Impact: Top 500 Inspections

In [ ]:
MONTHLY_INSPECTIONS = 500

print("=" * 80)
print(f"OPERATIONAL IMPACT: Top {MONTHLY_INSPECTIONS} Alerts per Month")
print("=" * 80)

for key, data in approaches.items():
    top_k_idx = np.argsort(data["y_prob"])[::-1][:MONTHLY_INSPECTIONS]
    hits = data["y_test"][top_k_idx].sum()
    precision_at_k = hits / MONTHLY_INSPECTIONS
    print(f"\n{data['name']}:")
    print(f"  Fraud found in top {MONTHLY_INSPECTIONS}: {int(hits)}/{MONTHLY_INSPECTIONS} ({precision_at_k:.1%} hit rate)")
    print(f"  Wasted inspections:  {MONTHLY_INSPECTIONS - int(hits)}")

## Key Takeaways

### Model Quality (ROC AUC — threshold-independent)
B (0.78) > A (0.76) > C (0.74). Enhanced features on raw consumption produce the best-ranked model overall. AUC improved across the board after adding early stopping and computed class weights.

### At F1-Optimal Threshold (the right comparison)
Default threshold (0.5) is suboptimal for all approaches. At each approach's F1-optimal threshold:
- **B leads** on F1 (0.36), Precision (0.34), and competitive FP count
- **A** is surprisingly strong baseline with new features (F1=0.33)
- **C** trades precision for recall — catches more thieves but with more false alarms

### The Zero-Shot Advantage (C's value proposition)
C's performance on this demo dataset understates its production value:
- **No retraining for new meters** — Chronos-2 is zero-shot, deployed once
- **Drift-resistant** — forecast adapts to changing consumption patterns automatically
- **Quantile-based anomaly detection** — actual vs prediction interval is a unique theft signal B cannot replicate
- With **hourly data + local holidays + real weather**, C's advantage grows

### What Gets Better with Real-World Data
- **Higher granularity** (hourly/15-min vs daily) = better forecasts = cleaner residuals
- **Real weather data** replaces simulated seasonality proxy
- **Local holiday calendar** = more relevant enrichment than generic calendars
- **Scale** (9.8M meters) = SageMaker endpoint autoscaling handles natively

---
*Demo built by AWS Specialist SA team. Chronos-2: zero-shot inference via SageMaker endpoint. XGBoost: trained on SageMaker AI using SDK v3. Early stopping and auto-calibrated class weights applied uniformly across all approaches.*

## Cleanup

In [ ]:
# Uncomment to delete S3 artifacts
# import subprocess
# sagemaker_session, _, bucket = get_sagemaker_session()
# subprocess.run(["aws", "s3", "rm", f"s3://{bucket}/ntl-detection-demo/", "--recursive"])
# print("S3 artifacts deleted")